# 🛒 AI Smart Price Finder

Real-time cheapest-price comparison across Google Shopping, Amazon, and more — powered by **Gemini AI**, **SerpAPI**, and **Streamlit**, tunneled live from Google Colab via **ngrok**.

> Run each cell below in order. Fill in your free API keys in **Cell 2** before running Cell 4.

**Tech stack:** Streamlit · SerpAPI · Google Gemini · Plotly · pyngrok


## CELL 1: Install all required packages

In [ ]:
import subprocess
import sys

def install_packages():
    print("⏳ Installing dependencies... Please wait.")
    packages = [
        "streamlit", "pyngrok", "google-genai", 
        "google-search-results", "requests", 
        "beautifulsoup4", "pandas", "plotly", "pillow"
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
    print('✅ All packages installed successfully!\n')

# Run installation logic
install_packages()

## CELL 2: Configure your API keys (Gemini Version)

In [ ]:
import os

# ⚠️  FILL THESE IN BEFORE RUNNING
SERPAPI_KEY    = "YOUR_SERPAPI_KEY_HERE"       # Get from https://serpapi.com (100 searches/month free)
GEMINI_KEY     = "YOUR_GEMINI_API_KEY_HERE"     # Get from https://aistudio.google.com (100% free tier)
NGROK_TOKEN    = "YOUR_NGROK_TOKEN_HERE"        # Get from https://dashboard.ngrok.com (Free)

os.environ["SERPAPI_KEY"] = SERPAPI_KEY
os.environ["GEMINI_KEY"]  = GEMINI_KEY
os.environ["NGROK_TOKEN"] = NGROK_TOKEN

print('✅ API keys configured!')
print(f'   SerpAPI  : {"✓ Set" if SERPAPI_KEY != "YOUR_SERPAPI_KEY_HERE" else "✗ NOT SET"}')
print(f'   Gemini AI: {"✓ Set" if GEMINI_KEY != "YOUR_GEMINI_API_KEY_HERE" else "✗ NOT SET"}')
print(f'   ngrok    : {"✓ Set" if NGROK_TOKEN != "YOUR_NGROK_TOKEN_HERE" else "✗ NOT SET"}\n')

## CELL 3: Write the full Streamlit application (Gemini Version)

In [ ]:
app_code = '''
import streamlit as st
import requests
import json
import os
import time
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from serpapi import GoogleSearch
from google import genai

# ── Page config ────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="AI Price Finder",
    page_icon="🛒",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── CSS Styling ────────────────────────────────────────────────────────────────
st.markdown("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
  
  html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
  
  .main { background: #0A0A0F; }
  
  .hero-banner {
    background: linear-gradient(135deg, #0F0F1A 0%, #1A0A2E 50%, #0A1628 100%);
    border: 1px solid rgba(100, 50, 255, 0.3);
    border-radius: 20px;
    padding: 40px;
    text-align: center;
    margin-bottom: 30px;
    position: relative;
    overflow: hidden;
  }
  .hero-banner::before {
    content: '';
    position: absolute; top: 0; left: 0; right: 0; bottom: 0;
    background: radial-gradient(ellipse at 50% 0%, rgba(100,50,255,0.15) 0%, transparent 70%);
    pointer-events: none;
  }
  .hero-title {
    font-size: 2.8rem; font-weight: 800;
    background: linear-gradient(135deg, #a78bfa, #60a5fa, #34d399);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    margin: 0; line-height: 1.2;
  }
  .hero-sub {
    color: rgba(255,255,255,0.55); font-size: 1.05rem;
    margin-top: 10px; font-weight: 400;
  }
  .badge-row {
    display: flex; gap: 10px; justify-content: center; margin-top: 20px; flex-wrap: wrap;
  }
  .badge {
    background: rgba(255,255,255,0.08);
    border: 1px solid rgba(255,255,255,0.12);
    border-radius: 20px; padding: 5px 14px;
    color: rgba(255,255,255,0.7); font-size: 0.78rem; font-weight: 500;
  }
  
  .product-card {
    background: linear-gradient(145deg, #12121F, #1A1A2E);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 16px; padding: 20px;
    transition: all 0.3s ease; margin-bottom: 16px;
    position: relative; overflow: hidden;
  }
  .product-card:hover {
    border-color: rgba(100,50,255,0.5);
    transform: translateY(-2px);
    box-shadow: 0 8px 32px rgba(100,50,255,0.15);
  }
  .product-card.best-deal {
    border-color: rgba(52,211,153,0.5);
    background: linear-gradient(145deg, #0D1F18, #1A2E24);
  }
  .rank-badge {
    position: absolute; top: 12px; right: 12px;
    border-radius: 8px; padding: 3px 10px;
    font-size: 0.7rem; font-weight: 700; letter-spacing: 0.05em;
  }
  .rank-1 { background: rgba(52,211,153,0.2); color: #34d399; border: 1px solid rgba(52,211,153,0.3); }
  .rank-2 { background: rgba(96,165,250,0.2); color: #60a5fa; border: 1px solid rgba(96,165,250,0.3); }
  .rank-3 { background: rgba(167,139,250,0.2); color: #a78bfa; border: 1px solid rgba(167,139,250,0.3); }
  .rank-n { background: rgba(255,255,255,0.05); color: rgba(255,255,255,0.4); border: 1px solid rgba(255,255,255,0.1); }
  
  .price-big {
    font-size: 1.8rem; font-weight: 800;
    background: linear-gradient(135deg, #34d399, #60a5fa);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
  }
  .source-name {
    font-size: 0.8rem; color: rgba(255,255,255,0.45);
    font-weight: 500; text-transform: uppercase; letter-spacing: 0.1em;
    margin-bottom: 6px;
  }
  .product-title {
    font-size: 0.95rem; font-weight: 600; color: rgba(255,255,255,0.9);
    margin: 6px 0; line-height: 1.4;
  }
  .rating-row { color: #facc15; font-size: 0.82rem; margin-top: 4px; }
  .delivery-tag {
    display: inline-block; margin-top: 8px;
    background: rgba(52,211,153,0.1); border: 1px solid rgba(52,211,153,0.2);
    color: #34d399; border-radius: 6px; padding: 2px 8px; font-size: 0.72rem;
  }
  
  .ai-insight {
    background: linear-gradient(135deg, #0F0A2A, #0A1628);
    border: 1px solid rgba(167,139,250,0.3);
    border-left: 4px solid #a78bfa;
    border-radius: 12px; padding: 24px;
    margin: 24px 0;
  }
  .ai-insight-title {
    font-size: 0.78rem; font-weight: 700; letter-spacing: 0.1em;
    color: #a78bfa; text-transform: uppercase; margin-bottom: 12px;
  }
  .ai-insight-body { color: rgba(255,255,255,0.8); font-size: 0.95rem; line-height: 1.7; }
  
  .stat-card {
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 12px; padding: 18px; text-align: center;
  }
  .stat-value { font-size: 1.6rem; font-weight: 800; color: #a78bfa; }
  .stat-label { font-size: 0.75rem; color: rgba(255,255,255,0.4); margin-top: 4px; text-transform: uppercase; }
  
  .search-history-item {
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.07);
    border-radius: 8px; padding: 10px 14px; margin-bottom: 8px;
    color: rgba(255,255,255,0.7); font-size: 0.85rem; cursor: pointer;
  }
  
  div[data-testid="stButton"] > button {
    background: linear-gradient(135deg, #6432ff, #4060ff) !important;
    color: white !important; border: none !important;
    border-radius: 10px !important; font-weight: 600 !important;
    padding: 0.6rem 1.5rem !important; width: 100%;
    transition: all 0.2s ease !important;
  }
  div[data-testid="stButton"] > button:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 6px 24px rgba(100,50,255,0.4) !important;
  }
  
  .stTextInput input, .stSelectbox select {
    background: rgba(255,255,255,0.06) !important;
    border: 1px solid rgba(255,255,255,0.12) !important;
    color: white !important; border-radius: 10px !important;
  }
  .stSlider > div { color: rgba(255,255,255,0.7) !important; }
  
  footer { visibility: hidden; }
  #MainMenu { visibility: hidden; }
</style>
""", unsafe_allow_html=True)

# ── Session state init ─────────────────────────────────────────────────────────
if "search_history" not in st.session_state:
    st.session_state.search_history = []
if "results_cache" not in st.session_state:
    st.session_state.results_cache = {}
if "current_results" not in st.session_state:
    st.session_state.current_results = None
if "ai_analysis" not in st.session_state:
    st.session_state.ai_analysis = ""

# ── API Keys from env ──────────────────────────────────────────────────────────
SERPAPI_KEY = os.environ.get("SERPAPI_KEY", "")
GEMINI_KEY  = os.environ.get("GEMINI_KEY", "")

# ─────────────────────────────────────────────────────────────────────────────
#  CORE FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def search_google_shopping(query, num_results=20, country="in", currency="INR"):
    """Search Google Shopping via SerpAPI."""
    try:
        params = {
            "engine": "google_shopping",
            "q": query,
            "api_key": SERPAPI_KEY,
            "num": num_results,
            "gl": country,
            "hl": "en"
        }
        search = GoogleSearch(params)
        results = search.get_dict()
        items = results.get("shopping_results", [])
        return items, None
    except Exception as e:
        return [], str(e)

def search_amazon_products(query, num_results=10):
    """Search Amazon via SerpAPI."""
    try:
        params = {
            "engine": "amazon",
            "k": query,
            "api_key": SERPAPI_KEY,
        }
        search = GoogleSearch(params)
        results = search.get_dict()
        items = results.get("organic_results", [])[:num_results]
        parsed = []
        for item in items:
            price_str = item.get("price", "")
            price_val = parse_price(price_str)
            if price_val:
                parsed.append({
                    "title":    item.get("title", "")[:80],
                    "price":    price_str,
                    "price_val": price_val,
                    "source":   "Amazon",
                    "link":     item.get("link", ""),
                    "rating":   item.get("rating", ""),
                    "reviews":  item.get("reviews", ""),
                    "thumbnail": item.get("thumbnail", ""),
                    "delivery": item.get("delivery", "")
                })
        return parsed
    except Exception:
        return []

def parse_price(price_str):
    """Extract numeric price from price string."""
    if not price_str:
        return None
    import re
    cleaned = re.sub(r"[^\d.]", "", str(price_str).replace(",", ""))
    try:
        val = float(cleaned)
        return val if val > 0 else None
    except:
        return None

def fetch_all_results(query, max_results, country):
    """Aggregate results from Google Shopping + Amazon."""
    all_items = []
    
    # Google Shopping results
    gshop_raw, err = search_google_shopping(query, num_results=max_results, country=country)
    for item in gshop_raw:
        price_str = item.get("price", "")
        price_val = parse_price(price_str)
        if price_val:
            source = item.get("source", "Unknown Store")
            all_items.append({
                "title":     item.get("title", "")[:80],
                "price":     price_str,
                "price_val": price_val,
                "source":    source,
                "link":      item.get("link", ""),
                "rating":    item.get("rating", ""),
                "reviews":   item.get("reviews", ""),
                "thumbnail": item.get("thumbnail", ""),
                "delivery":  item.get("delivery", "")
            })
    
    # Amazon results
    amazon_items = search_amazon_products(query, num_results=8)
    all_items.extend(amazon_items)
    
    # Remove dupes by (source, price)
    seen = set()
    unique = []
    for item in all_items:
        key = (item["source"].lower(), item["price_val"])
        if key not in seen:
            seen.add(key)
            unique.append(item)
    
    # Sort by price ascending
    unique.sort(key=lambda x: x["price_val"])
    return unique

def get_ai_analysis(query, results):
    \"\"\"Get Google Gemini recommendation on the search results.\"\"\"
    GEMINI_KEY = os.environ.get("GEMINI_KEY", "")
    if not GEMINI_KEY or not results:
        return ""
    try:
        client = genai.Client(api_key=GEMINI_KEY)
        top5 = results[:5]
        products_text = "\\n".join([
            f"{i+1}. {p['title']} | {p['price']} | {p['source']} | Rating: {p.get('rating','N/A')} ({p.get('reviews','')} reviews) | Delivery: {p.get('delivery','unknown')}"
            for i, p in enumerate(top5)
        ])
        prices = [p["price_val"] for p in results]
        avg_price = sum(prices) / len(prices)
        min_price = min(prices)
        max_price = max(prices)
        
        prompt = f\"\"\"You are a smart shopping assistant. A user is searching for: \\\"{query}\\\"\n
\n
Here are the top 5 cheapest results found across shopping sites:\n
{products_text}\n
\n
Price stats across {len(results)} results:\n
- Cheapest: {min_price:.2f}\n
- Most expensive: {max_price:.2f}  \n
- Average: {avg_price:.2f}\n
\n
In 3-4 sentences, give a sharp buying recommendation:\n
1. Which deal is the best value and why\n
2. Any red flags or things to watch (delivery cost, low ratings, unfamiliar sellers)\n
3. One practical tip for this specific product category\n
\n
Be direct, specific, and genuinely helpful. No fluff.\"\"\"
        
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
        )
        return response.text
    except Exception as e:
        return f"AI analysis unavailable: {str(e)}"

def render_stars(rating):
    """Convert numeric rating to star display."""
    try:
        r = float(str(rating))
        full = int(r)
        half = 1 if (r - full) >= 0.5 else 0
        empty = 5 - full - half
        return "★" * full + ("☆" if half else "") + "☆" * empty + f" {r:.1f}"
    except:
        return ""

# ─── SIDEBAR ─────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### ⚙️ Search Settings")
    
    country = st.selectbox(
        "🌍 Country / Region",
        options=["in", "us", "gb", "de", "au", "ca", "sg"],
        format_func=lambda x: {"in":"🇮🇳 India","us":"🇺🇸 USA","gb":"🇬🇧 UK",
                                "de":"🇩🇪 Germany","au":"🇦🇺 Australia",
                                "ca":"🇨🇦 Canada","sg":"🇸🇬 Singapore"}.get(x,x),
        index=0
    )
    
    max_results = st.slider("📦 Max Results to Fetch", 10, 50, 20, 5)
    
    show_charts = st.toggle("📊 Show Price Charts", value=True)
    use_ai = st.toggle("🤖 AI Recommendation", value=True)
    
    st.markdown("---")
    st.markdown("### 🔍 Quick Category Searches")
    quick_searches = [
        "iPhone 15", "Samsung Galaxy S24", "boAt headphones",
        "laptop under 50000", "mechanical keyboard", "gaming mouse",
        "protein powder 1kg", "running shoes", "air purifier"
    ]
    for qs in quick_searches:
        if st.button(qs, key=f"quick_{qs}"):
            st.session_state["quick_search"] = qs
    
    st.markdown("---")
    if st.session_state.search_history:
        st.markdown("### 🕘 Recent Searches")
        for h in reversed(st.session_state.search_history[-5:]):
            st.markdown(f"`{h}`")

# ─── HERO BANNER ─────────────────────────────────────────────────────────────
st.markdown("""
<div class="hero-banner">
  <div class="hero-title">🛒 AI Smart Price Finder</div>
  <div class="hero-sub">Real-time cheapest prices across Amazon, Flipkart, and 100+ stores • Powered by Gemini AI</div>
  <div class="badge-row">
    <span class="badge">🔴 Live Prices</span>
    <span class="badge">🤖 AI Analysis</span>
    <span class="badge">📊 Price Charts</span>
    <span class="badge">🌍 Multi-country</span>
    <span class="badge">⚡ Google Shopping</span>
  </div>
</div>
""", unsafe_allow_html=True)

# ─── SEARCH BAR ─────────────────────────────────────────────────────────────
col_search, col_btn = st.columns([5, 1])

default_query = st.session_state.pop("quick_search", "") if "quick_search" in st.session_state else ""

with col_search:
    query = st.text_input(
        "🔍 What are you looking for?",
        value=default_query,
        placeholder="e.g. boAt Airdopes 141, Samsung TV 55 inch, Nike Air Max ...",
        label_visibility="collapsed"
    )

with col_btn:
    search_clicked = st.button("🔍 Search", use_container_width=True)

# ─── SEARCH EXECUTION ─────────────────────────────────────────────────────────────
if search_clicked and query.strip():
    if not SERPAPI_KEY or SERPAPI_KEY == "YOUR_SERPAPI_KEY_HERE":
        st.error("❌ Please set your SERPAPI_KEY in Cell 2 of the Colab notebook!")
        st.stop()
    
    with st.spinner(f"🔍 Scanning {max_results} listings across stores in real-time..."):
        t_start = time.time()
        results = fetch_all_results(query.strip(), max_results, country)
        elapsed = time.time() - t_start
    
    if not results:
        st.warning("😕 No results found. Try a different search term.")
        st.stop()
    
    st.session_state.current_results = results
    st.session_state.current_query   = query.strip()
    st.session_state.search_elapsed  = elapsed
    
    if query.strip() not in st.session_state.search_history:
        st.session_state.search_history.append(query.strip())
    
    if use_ai:
        with st.spinner("🤖 Google Gemini is analysing deals..."):
            st.session_state.ai_analysis = get_ai_analysis(query.strip(), results)
    else:
        st.session_state.ai_analysis = ""

# ─── RESULTS DISPLAY ─────────────────────────────────────────────────────────────
if st.session_state.current_results:
    results = st.session_state.current_results
    query_text = st.session_state.get("current_query", "")
    elapsed = st.session_state.get("search_elapsed", 0)
    
    prices = [r["price_val"] for r in results]
    sources = list(set([r["source"] for r in results]))
    savings = max(prices) - min(prices)
    
    # ── Stats Row ─────────────────────────────────────────────────────────────
    st.markdown(f"### 📊 Results for: *{query_text}* — {len(results)} deals found in {elapsed:.1f}s")
    
    c1, c2, c3, c4 = st.columns(4)
    with c1:
        st.markdown(f\"\"\"
        <div class="stat-card">
          <div class="stat-value">{results[0]["price"]}</div>
          <div class="stat-label">Best Price 🏆</div>
        </div>\"\"\", unsafe_allow_html=True)
    with c2:
        avg = sum(prices)/len(prices)
        st.markdown(f\"\"\"
        <div class="stat-card">
          <div class="stat-value">{avg:,.0f}</div>
          <div class="stat-label">Avg Price</div>
        </div>\"\"\", unsafe_allow_html=True)
    with c3:
        st.markdown(f\"\"\"
        <div class="stat-card">
          <div class="stat-value">{len(sources)}</div>
          <div class="stat-label">Stores Found</div>
        </div>\"\"\", unsafe_allow_html=True)
    with c4:
        st.markdown(f\"\"\"
        <div class="stat-card">
          <div class="stat-value">{savings:,.0f}</div>
          <div class="stat-label">Max Savings 💰</div>
        </div>\"\"\", unsafe_allow_html=True)
    
    st.markdown("<br>", unsafe_allow_html=True)
    
    # ── AI Analysis ───────────────────────────────────────────────────────────
    if st.session_state.ai_analysis:
        st.markdown(f\"\"\"
        <div class="ai-insight">
          <div class="ai-insight-title">🤖 Gemini AI Recommendation</div>
          <div class="ai-insight-body">{st.session_state.ai_analysis}</div>
        </div>\"\"\", unsafe_allow_html=True)
    
    # ── Price Chart ───────────────────────────────────────────────────────────
    if show_charts and len(results) > 3:
        st.markdown("#### 📈 Price Comparison Chart")
        
        chart_data = results[:15]
        df = pd.DataFrame(chart_data)
        df["label"] = df.apply(lambda r: f"{r['source']}\\n{r['price']}", axis=1)
        
        colors = ["#34d399" if i == 0 else "#6432ff" if i < 3 else "#4060ff" 
                  for i in range(len(df))]
        
        fig = go.Figure(go.Bar(
            x=df["source"],
            y=df["price_val"],
            text=df["price"],
            textposition="outside",
            marker_color=colors,
            hovertemplate="<b>%{x}</b><br>Price: %{text}<br>Title: %{customdata}<extra></extra>",
            customdata=df["title"]
        ))
        fig.update_layout(
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(255,255,255,0.03)",
            font=dict(color="rgba(255,255,255,0.7)", family="Inter"),
            margin=dict(l=20, r=20, t=20, b=40),
            height=320,
            xaxis=dict(gridcolor="rgba(255,255,255,0.05)", tickangle=-30),
            yaxis=dict(gridcolor="rgba(255,255,255,0.05)"),
            showlegend=False
        )
        st.plotly_chart(fig, use_container_width=True)
    
    # ── Product Cards ─────────────────────────────────────────────────────────
    st.markdown("#### 🛍️ Sorted by Price (Cheapest First)")
    
    for i, item in enumerate(results):
        rank_class = ["rank-1","rank-2","rank-3"][i] if i < 3 else "rank-n"
        rank_label = ["🥇 BEST DEAL","🥈 2nd","🥉 3rd"][i] if i < 3 else f"#{i+1}"
        card_class  = "product-card best-deal" if i == 0 else "product-card"
        
        stars = render_stars(item.get("rating",""))
        delivery_html = f'<span class="delivery-tag">🚚 {item["delivery"]}</span>' if item.get("delivery") else ""
        reviews_text  = f'({item["reviews"]} reviews)' if item.get("reviews") else ""
        link_html = f'<a href="{item["link"]}" target="_blank" style="color:#60a5fa;font-size:0.82rem;text-decoration:none;">🔗 View Product</a>' if item.get("link") else ""
        
        col_thumb, col_info = st.columns([1, 6])
        with col_thumb:
            if item.get("thumbnail"):
                st.image(item["thumbnail"], width=80)
        with col_info:
            st.markdown(f\"\"\"
            <div class="{card_class}">
              <span class="rank-badge {rank_class}">{rank_label}</span>
              <div class="source-name">{item['source']}</div>
              <div class="product-title">{item['title']}</div>
              <div class="price-big">{item['price']}</div>
              <div class="rating-row">{stars} {reviews_text}</div>
              {delivery_html}
              <div style="margin-top:10px;">{link_html}</div>
            </div>\"\"\", unsafe_allow_html=True)
    
    # ── Download as CSV ───────────────────────────────────────────────────────
    st.markdown("---")
    df_export = pd.DataFrame([{
        "Rank":    i+1,
        "Title":   r["title"],
        "Price":   r["price"],
        "Source":  r["source"],
        "Rating":  r.get("rating",""),
        "Reviews": r.get("reviews",""),
        "Link":    r.get("link","")
    } for i,r in enumerate(results)])
    
    csv_data = df_export.to_csv(index=False).encode("utf-8")
    col_dl1, col_dl2 = st.columns([1,3])
    with col_dl1:
        st.download_button(
            label="⬇️ Download Results CSV",
            data=csv_data,
            file_name=f"price_comparison_{query_text.replace(' ','_')}.csv",
            mime="text/csv"
        )

else:
    # ── Welcome state ─────────────────────────────────────────────────────────
    st.markdown("")
    c1, c2, c3 = st.columns(3)
    cards = [
        ("🔴", "Real-time Prices", "Live data from Google Shopping and Amazon — no cached results"),
        ("🤖", "AI Recommendations", "Gemini AI analyses deals and tells you the smartest buy"),
        ("📊", "Visual Comparison", "Interactive charts so you can see price gaps at a glance")
    ]
    for col, (icon, title, desc) in zip([c1,c2,c3], cards):
        with col:
            st.markdown(f\"\"\"
            <div class="stat-card" style="text-align:left;padding:24px;">
              <div style="font-size:2rem;margin-bottom:12px;">{icon}</div>
              <div style="font-weight:700;color:white;margin-bottom:8px;">{title}</div>
              <div style="color:rgba(255,255,255,0.5);font-size:0.85rem;line-height:1.5;">{desc}</div>
            </div>\"\"\", unsafe_allow_html=True)
    
    st.markdown("<br><div style='text-align:center;color:rgba(255,255,255,0.3);font-size:0.85rem;'>👆 Type a product name above and click Search to get started</div>", unsafe_allow_html=True)
'''

with open("/content/price_finder_app.py", "w") as f:
    f.write(app_code)

print('✅ Streamlit app written to /content/price_finder_app.py\n')

## CELL 4: Launch Streamlit + ngrok

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok, conf

# Authenticate ngrok
conf.get_default().auth_token = os.environ.get("NGROK_TOKEN", "")

# Kill any existing background instances
print("🧹 Cleaning up stale background processes...")
subprocess.call("pkill -f streamlit 2>/dev/null || true", shell=True)
subprocess.call("pkill -f ngrok 2>/dev/null || true", shell=True)
time.sleep(2)

# Start Streamlit framework in standard background pipe
print("⏳ Starting Streamlit instance...")
proc = subprocess.Popen(
    ["streamlit", "run", "/content/price_finder_app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--theme.base=dark",
     "--theme.backgroundColor=#0A0A0F",
     "--theme.secondaryBackgroundColor=#12121F",
     "--theme.textColor=#FFFFFF",
     "--theme.primaryColor=#6432ff"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait execution buffer for boot routines
time.sleep(5)

# Spin up tunneling server forward pipeline map
tunnel = ngrok.connect(8501)
public_url = tunnel.public_url

print("\n" + "═"*60)
print("  🛒  AI SMART PRICE FINDER — LIVE COMPILATION SUCCESS!")
print("═"*60)
print(f"  🌐  Public Tunnel URL : {public_url}")
print(f"  🏠  Local Network URL : http://localhost:8501")
print("═"*60)
print("  ✅ App compilation active. Launch the Public Tunnel URL above.")
print("  ⏹  Press Ctrl+C or trigger execution termination steps to halt application runtime.")
print("═"*60 + "\n")

# Process watch stay alive block routine
try:
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    ngrok.kill()
    print("\n🛑 Execution routine interrupted. App services stopped.")